In [ ]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

SEED = 1   # makes results the same every run

## Part 1:

### Part 1(a): Percentage of late orders

#### A late order is one whose delivery time is bigger than the promise. The plan is to collect the delivery times for every zone and time block requested, then count how many are late.

In [22]:
def late_rate(zone, time_block, promise):
    # Step 1: decide which zones and time blocks to look at

    zones = ZONES if zone == "all" else [zone]
    time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    # Step 2: count orders and late orders

    total_orders = 0
    late_orders = 0

    for z in zones:
        for t in time_blocks:
            times = delivery_times(z, t, promise, seed=42)

            total_orders = total_orders + len(times)
            late_orders = late_orders + (times > promise).sum()

    # Step 3: calculate percentage

    return late_orders / total_orders * 100


Testing the 4 required cases at 45 mintue promise

In [29]:
print("1. Specific zone + specific time block")
print(f"Far West + Fri/Sat eve: {late_rate('Far West', 'Fri/Sat eve', 45):.2f}%")

print("\n2. All zones + specific time block")
print(f"All zones + Lunch: {late_rate('all', 'Lunch', 45):.2f}%")

print("\n3. Specific zone + all time blocks")
print(f"Central + all time blocks: {late_rate('Central', 'all', 45):.2f}%")

print("\n4. All zones + all time blocks")
print(f"All zones + all time blocks: {late_rate('all', 'all', 45):.2f}%")

1. Specific zone + specific time block
Far West + Fri/Sat eve: 36.67%

2. All zones + specific time block
All zones + Lunch: 3.68%

3. Specific zone + all time blocks
Central + all time blocks: 5.36%

4. All zones + all time blocks
All zones + all time blocks: 9.29%


### Part 1 (b): Rank by late rate

In [30]:
late_rates = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:
        rate = late_rate(zone, time_block, PROMISE)
        late_rates.append((zone, time_block, rate))

late_rates.sort(key=lambda x: x[2], reverse=True)

print("Rank by late rate:")
for zone, time_block, rate in late_rates:
    print(zone, "-", time_block, ":", round(rate, 2), "%")

Rank by late rate:
Far West - Fri/Sat eve : 36.67 %
North - Fri/Sat eve : 18.0 %
Far West - Weekday eve : 10.59 %
North - Weekday eve : 9.2 %
Central - Fri/Sat eve : 7.89 %
Central - Weekday eve : 6.97 %
Far West - Other : 6.25 %
North - Lunch : 4.21 %
Central - Lunch : 3.85 %
North - Other : 2.5 %
Far West - Lunch : 2.5 %
Central - Other : 2.11 %


### Part 1 (c): Average delivery time

In [31]:
def avg_time(zone, time_block, promise):

    # Step 1: decide which zones and time blocks to look at
    zones = ZONES if zone == "all" else [zone]
    time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    # Step 2: count orders and minutes across all of them
    total_orders = 0
    total_minutes = 0

    for z in zones:
        for t in time_blocks:
            times = delivery_times(z, t, promise, seed=42)

            total_orders = total_orders + len(times)
            total_minutes = total_minutes + times.sum()

    # Step 3: Calculate average delivery time
    return total_minutes / total_orders


Testing the 4 required cases with 45 minute promise:

In [32]:
print("1. Specific zone + specific time block")
print(f"Far West + Fri/Sat eve: {avg_time('Far West', 'Fri/Sat eve', 45):.2f} minutes")

print("\n2. All zones + specific time block")
print(f"All zones + Lunch: {avg_time('all', 'Lunch', 45):.2f} minutes")

print("\n3. Specific zone + all time blocks")
print(f"Central + all time blocks: {avg_time('Central', 'all', 45):.2f} minutes")

print("\n4. All zones + all time blocks")
print(f"All zones + all time blocks: {avg_time('all', 'all', 45):.2f} minutes")

1. Specific zone + specific time block
Far West + Fri/Sat eve: 42.98 minutes

2. All zones + specific time block
All zones + Lunch: 28.49 minutes

3. Specific zone + all time blocks
Central + all time blocks: 29.01 minutes

4. All zones + all time blocks
All zones + all time blocks: 31.56 minutes


### Part 1 (d): Rank by Average time

In [33]:
average_times = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:
        average = avg_time(zone, time_block, PROMISE)
        average_times.append((zone, time_block, average))

average_times.sort(key=lambda x: x[2], reverse=True)

print("Rank by average delivery time:")
for zone, time_block, average in average_times:
    print(zone, "-", time_block, ":", round(average, 2), "minutes")

Rank by average delivery time:
Far West - Fri/Sat eve : 42.98 minutes
North - Fri/Sat eve : 37.07 minutes
Far West - Weekday eve : 35.25 minutes
Central - Fri/Sat eve : 32.39 minutes
North - Weekday eve : 31.94 minutes
Far West - Lunch : 30.37 minutes
Far West - Other : 30.33 minutes
Central - Weekday eve : 29.98 minutes
North - Lunch : 28.91 minutes
North - Other : 27.94 minutes
Central - Lunch : 27.32 minutes
Central - Other : 26.24 minutes


### Part 1(e): Which measure matters more?

- The late rate is more relevant to Rosa. Rosa's costs come from late orders: each one triggers a refund and makes the customer order less in the future. The promise is 45 minutes, so a customer only cares whether the pizza arrived within 45 minutes, not whether the average is 30.

- The average hides the problem. Far West Fri/Sat eve has an average of 42.9 minutes, which is under 45 and looks fine, but 36.7% of its orders are late. A few very slow deliveries push many orders past the promise while the average still looks okay.

- The two rankings also disagree in the middle. Far West Lunch is 6th by average time (30.6 min) but has only a 2.5% late rate, tied for the lowest. Far West "Other" has a higher late rate (4.7%) but a lower average. So the average would send Rosa's attention to the wrong places. The average is still useful as a secondary signal, because slower averages mean orders are closer to being late.

## Part 2: Best Promise

### Part 2(a): Cost per late order

In [ ]:
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [ ]:
def cost_per_late_order(costs):
    return costs["refund"] + costs["churn_orders"] * costs["margin"]

print("Total cost per late order: $", cost_per_late_order(COSTS))

Total cost per late order: $ 26.2


### Part 2(b): Choosing the range

In [ ]:
# From part a
def cost_per_late_order(costs):
    return costs["refund"] + costs["churn_orders"] * costs["margin"]

# Part II b
def best_promise(zone, time_block, promises, costs):
    late_cost = cost_per_late_order(costs)

    best_promise_so_far = None
    best_profit_so_far = None

    for p in promises:
        # Get every delivery time for this promise
        times = delivery_times(zone, time_block, p, seed=SEED)

        number_of_orders = len(times)
        number_of_late = (times > p).sum()

        # Net profit = money from all orders - cost of late orders
        profit = number_of_orders * costs["margin"] - number_of_late * late_cost

        # Keep this promise if it beats the best one so far
        if best_profit_so_far is None or profit > best_profit_so_far:
            best_promise_so_far = p
            best_profit_so_far = profit

    return best_promise_so_far, best_profit_so_far

Test it:

In [21]:
promises_to_try = list(range(20, 85, 5))   # 20, 25, 30, ..., 80

p, profit = best_promise("Far West", "Fri/Sat eve", promises_to_try, COSTS)
print("Recommended promise:", p, "minutes")
print("Net profit: $", round(profit, 2))

Recommended promise: 55 minutes
Net profit: $ 1212.4


Why range 20 to 80:

- A short promise gets more orders, but almost all of them are late, so profit is negative. Around 20 minutes is already clearly a loss, so there's no need to go lower.

- A long promise has almost no late orders, but customers order less, so profit falls again. At 65+ minutes for Far West Fri/Sat eve, there are almost no late orders, and profit only shrinks from there.

- The range must go past 50. If I had stopped at 50, I would have picked 50 minutes ($827). The true best is 55 minutes ($1,212), which would have been missed.

- General rule: choose a range wide enough that the best answer lands in the middle, not at the edge. If the winner is the first or last value tried, widen the range and try again.